# 00 · Entorno y formato del artefacto

**Qué hace:** comprueba las versiones del entorno frente a las que fija `rc_pronostico` y verifica que **skops** sirve como formato del artefacto (D-16): que guarda y carga un `HistGradientBoostingRegressor` cuantil con variables categóricas, que predice igual, que solo acepta tipos de confianza y que un byte alterado se rechaza por SHA-256.

**Decisiones que aplica:** D-16 (formato y seguridad), D-19 (entorno propio), D-18 (reproducibilidad).

**Deja:** nada en disco (todo se hace en un directorio temporal). Si skops falla, se anota en D-16 y se usa joblib con huella obligatoria.

In [1]:
# Parámetros
SEMILLA = 2026
CUANTILES = (0.5, 0.9)

## 1. Versiones frente a las fijadas por `rc_pronostico`

In [2]:
import sys, platform, hashlib, tempfile, tomllib
from importlib import metadata
from pathlib import Path
import numpy as np, pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "cuadernos" else Path.cwd()
req = tomllib.loads((RAIZ / "rc_pronostico" / "pyproject.toml").read_text(encoding="utf-8"))["project"]
fijadas = {r.split("==")[0]: r.split("==")[1] for r in req["dependencies"] if "==" in r}

print("Python", platform.python_version(), "| SO", platform.platform())
filas = []
for paquete in ["numpy", "pandas", "scikit-learn", "skops"]:
    instalada = metadata.version(paquete)
    fija = fijadas.get(paquete, "(libre)")
    filas.append((paquete, instalada, fija, fija == "(libre)" or fija == instalada))
tabla = pd.DataFrame(filas, columns=["paquete", "instalada", "fijada", "coincide"])
tabla

Python 3.12.10 | SO Windows-11-10.0.26200-SP0


,paquete,instalada,fijada,coincide
0,numpy,2.5.5,(libre),True
1,pandas,3.1.1,(libre),True
2,scikit-learn,1.7.2,1.7.2,True
3,skops,0.13.0,0.13.0,True


`scikit-learn` y `skops` se fijan con `==` porque el artefacto solo carga con la misma versión (CONTRATO, 5). `numpy` y `pandas` quedan libres en el paquete, pero se registran en la ficha del modelo.

## 2. Prueba de skops con un HistGradientBoosting cuantil y categóricas

In [3]:
import skops.io as sio
from sklearn.ensemble import HistGradientBoostingRegressor

rng = np.random.default_rng(SEMILLA)
n = 600
X = pd.DataFrame({
    "producto": rng.integers(0, 7, n),          # código de categoría
    "dia_semana": rng.integers(0, 7, n),
    "lag_7": rng.poisson(20, n).astype(float),
})
X.loc[rng.random(n) < 0.05, "lag_7"] = np.nan   # NaN = día cerrado / sin dato
y = 5 + 3 * X["dia_semana"] + X["producto"] + rng.normal(0, 2, n)
categoricas = [True, True, False]

modelos = {q: HistGradientBoostingRegressor(
    loss="quantile", quantile=q, categorical_features=categoricas,
    early_stopping=False, random_state=SEMILLA, max_iter=50).fit(X, y) for q in CUANTILES}
{q: round(float(m.predict(X).mean()), 3) for q, m in modelos.items()}

{0.5: 17.012, 0.9: 19.338}

In [4]:
tmp = Path(tempfile.mkdtemp(prefix="rc_skops_"))
archivo = tmp / "modelo.skops"
paquete = {"p50": modelos[0.5], "p90": modelos[0.9],
           "codigos_producto": {f"prod-{i}": i for i in range(7)}}
sio.dump(paquete, archivo)
sha256 = hashlib.sha256(archivo.read_bytes()).hexdigest()

# Qué tipos no son de skops/numpy/sklearn: son los que habría que "confiar" explícitamente.
no_confiables = sio.get_untrusted_types(file=archivo)
print("Tamaño:", archivo.stat().st_size, "bytes | SHA-256:", sha256[:16], "…")
print("Tipos fuera de la lista segura:", no_confiables)

Tamaño: 932975 bytes | SHA-256: 36ae90c2334ae00d …
Tipos fuera de la lista segura: ['functools.partial', 'sklearn._loss.link.IdentityLink', 'sklearn._loss.link.Interval', 'sklearn._loss.loss.PinballLoss', 'sklearn.ensemble._hist_gradient_boosting.binning._BinMapper', 'sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor', 'sklearn.utils.validation.check_array']


In [5]:
# Carga solo con tipos de confianza (trusted=[] => no se acepta ningún tipo extra)
cargado = sio.load(archivo, trusted=no_confiables)
igual = all(np.array_equal(cargado[k].predict(X), paquete[k].predict(X)) for k in ("p50", "p90"))
print("Predicciones idénticas tras guardar y cargar:", igual)

Predicciones idénticas tras guardar y cargar: True


## 3. Rechazos: tipo no confiable y archivo alterado

In [6]:
# (a) Un tipo no permitido se rechaza.
class Intruso:  # simula un objeto con código arbitrario
    pass

hostil = tmp / "hostil.skops"
sio.dump({"x": Intruso()}, hostil)
try:
    sio.load(hostil, trusted=[])
    rechaza_tipo = False
except Exception as e:
    rechaza_tipo = True
    print("Tipo no confiable rechazado:", type(e).__name__)

# (b) Un byte alterado no pasa la verificación de huella (se verifica ANTES de cargar).
def cargar_verificado(ruta, sha_esperado, trusted):
    real = hashlib.sha256(Path(ruta).read_bytes()).hexdigest()
    if real != sha_esperado:
        raise ValueError(f"SHA-256 no coincide: {real[:12]} != {sha_esperado[:12]}")
    return sio.load(ruta, trusted=trusted)

alterado = tmp / "alterado.skops"
datos = bytearray(archivo.read_bytes()); datos[len(datos) // 2] ^= 0x01
alterado.write_bytes(bytes(datos))
try:
    cargar_verificado(alterado, sha256, no_confiables)
    rechaza_byte = False
except ValueError as e:
    rechaza_byte = True
    print("Byte alterado rechazado:", e)
cargar_verificado(archivo, sha256, no_confiables) is not None

Tipo no confiable rechazado: UntrustedTypesFoundException
Byte alterado rechazado: SHA-256 no coincide: 6fd0fb3f9382 != 36ae90c2334a


True

## Verificaciones

In [7]:
assert tabla.loc[tabla.paquete.isin(fijadas), "coincide"].all(), "scikit-learn/skops instalados no coinciden con rc_pronostico"
assert platform.python_version_tuple()[:2] == ("3", "12")
assert igual, "skops no reproduce las predicciones"
assert rechaza_tipo, "skops aceptó un tipo no confiable"
assert rechaza_byte, "un byte alterado no fue detectado"
print("OK · skops es viable como formato del artefacto (D-16 resuelta)")

OK · skops es viable como formato del artefacto (D-16 resuelta)
